In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from rich import print
from langchain_core.tools import tool   ## hat pair hame tool deta hai
from langchain_core.messages import HumanMessage , ToolMessage  ## tool ka massage show karne ke liye hamne ToolMessage use kiya hai
from langchain.chat_models import init_chat_model           ## large model ko call karne ke liye use kiya hai

In [85]:
load_dotenv()
MODEL = os.getenv("GROQ_MODELS" , "groq:qwen/qwen3.8-27b")

In [3]:
MODEL

'groq:qwen/qwen3.8-27b'

# Path of the resume folder

In [20]:
RESUME_DIR = (Path().cwd().parent / "resumes")
RESUME_DIR

WindowsPath('f:/agentic_ai_course_v2_live/season-1-foundations/ep_04_tool_calling.py/resumes')

In [6]:
def add(a: float , b:float) -> float :
    return a + b 
    

#### halusiniation : -  ham tb bolte hai jb bhi hmara llm se answer glt ata hai . use ham halusiniation bolte hai.

In [7]:
add(3.9, 11.0)

14.9

## Use Tools

In [8]:
@tool
def add(a: float , b:float) -> float :
    """ Add function use for the addtion of two number """
    return a + b 

In [9]:
add

StructuredTool(name='add', description='Add function use for the addtion of two number', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x00000216BA1954E0>)

In [11]:
def get_weather(city : str) ->str:
    """ Get the current weather of given city"""
    weather = {"mumbai" : "32c , humid" , "delhi" : "38c , haty" , "bangaluru" : "26c ,plesent"}
    return f"City {city} - {weather.get(city.lower() , f"No Weather data for {city}")}"


In [12]:
get_weather("mumbai")

'City mumbai - 32c , humid'

In [13]:
get_weather("pune")

'City pune - No Weather data for pune'

## Use Tool

In [14]:
@tool
def get_weather(city : str) ->str:
    """ Get the current weather of given city"""
    weather = {"mumbai" : "32c , humid" , "delhi" : "38c , haty" , "bangaluru" : "26c ,plesent"}
    return f"City {city} - {weather.get(city.lower() , f"No Weather data for {city}")}"


In [15]:
get_weather

StructuredTool(name='get_weather', description='Get the current weather of given city', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x00000216BB764900>)

In [16]:
get_weather.name

'get_weather'

In [17]:
get_weather.description

'Get the current weather of given city'

In [18]:
def lookup_user(user_id: str) -> str:
    """Look up a user's and plan by their id (e.g. 'u1')."""
    users = {"u1" : "Arti (Pro plan)", "u2" : "Diya (Free plan)"}
    return users.get(user_id , f"No user with id '{user_id}'.")


## Use Tool

In [19]:
@tool
def lookup_user(user_id:str)->str:
    """Look up a user and plan by their id (i.e,'u1')."""
    users = {"u1" : "Arti (pro plan)" , "u2" : "Diya (free plan)"}
    return users.get(user_id , f"No user with id '{user_id}'.")


In [21]:
def get_resume(name: str)->str:
    """ Read and return store resume for candidate name (ramesh ,khushi) """
    file_name = name.lower().strip().split()[0]
    file_path = RESUME_DIR / f"{name}.txt"
    if file_path.exists():
        return file_path.read_text(encoding='utf-8')
    return f"No resume found for {name}"

In [22]:
get_resume("priya")

'Name: Priya Singh\nRole: DevOps Engineer\nExperience: 5 years\nSkills: AWS, Kubernetes, Docker, Terraform, CI/CD (GitHub Actions), Linux\nBackground:\n5 years of hands-on experience in cloud infrastructure and automating deployment pipelines. Expert in AWS services, managing Kubernetes clusters, and infrastructure-as-code with Terraform.'

In [23]:
print(get_resume("priya"))

Name: Priya Singh
Role: DevOps Engineer
Experience: 5 years
Skills: AWS, Kubernetes, Docker, Terraform, CI/CD (GitHub Actions), Linux
Background:
5 years of hands-on experience in cloud infrastructure and automating deployment pipelines. Expert in AWS services,
managing Kubernetes clusters, and infrastructure-as-code with Terraform.

In [24]:
get_resume("arti")

'No resume found for arti'

## Use Tool

In [25]:
@tool
def get_resume(name:str)->str:
    """ Read and return store resume for candidate name (ramesh ,khushi) """
    file_name = name.lower().strip().split()[0]
    file_path = RESUME_DIR / f"{name}.txt"
    if file_path.exists():
        return file_path.read_text(encoding="utf-8")
    return f"No resume found for {name}"

In [26]:
get_resume

StructuredTool(name='get_resume', description='Read and return store resume for candidate name (ramesh ,khushi)', args_schema=<class 'langchain_core.utils.pydantic.get_resume'>, func=<function get_resume at 0x00000216BBA84040>)

In [27]:
TOOLS = [add ,get_weather , lookup_user , get_resume]


In [28]:
TOOLS

[StructuredTool(name='add', description='Add function use for the addtion of two number', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x00000216BA1954E0>),
 StructuredTool(name='get_weather', description='Get the current weather of given city', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x00000216BB764900>),
 StructuredTool(name='lookup_user', description="Look up a user and plan by their id (i.e,'u1').", args_schema=<class 'langchain_core.utils.pydantic.lookup_user'>, func=<function lookup_user at 0x00000216BB8D5B20>),
 StructuredTool(name='get_resume', description='Read and return store resume for candidate name (ramesh ,khushi)', args_schema=<class 'langchain_core.utils.pydantic.get_resume'>, func=<function get_resume at 0x00000216BBA84040>)]

In [29]:
TOOLS_DATA = {
                "add" : add , 
                "get_weather" : get_weather ,
                "lookup_user" : lookup_user ,
                "get_resume" : get_resume
            }

In [30]:
TOOLS_DATA

{'add': StructuredTool(name='add', description='Add function use for the addtion of two number', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x00000216BA1954E0>),
 'get_weather': StructuredTool(name='get_weather', description='Get the current weather of given city', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x00000216BB764900>),
 'lookup_user': StructuredTool(name='lookup_user', description="Look up a user and plan by their id (i.e,'u1').", args_schema=<class 'langchain_core.utils.pydantic.lookup_user'>, func=<function lookup_user at 0x00000216BB8D5B20>),
 'get_resume': StructuredTool(name='get_resume', description='Read and return store resume for candidate name (ramesh ,khushi)', args_schema=<class 'langchain_core.utils.pydantic.get_resume'>, func=<function get_resume at 0x00000216BBA84040>)}

In [31]:
TOOLS_DATA["add"]

StructuredTool(name='add', description='Add function use for the addtion of two number', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x00000216BA1954E0>)

### tool ko connect karne ke liye hum (.bind_tools ) use karte hai

In [37]:
llm = init_chat_model(MODEL , temperature=0)
llm_with_tools = llm.bind_tools(TOOLS)

In [53]:
prompt="can we select priya for Python developer interview"
result = llm_with_tools.invoke(prompt)

In [54]:
result

AIMessage(content="I don't have any information about a candidate named Priya in my available tools. Let me check if there's a resume available for her.\n\n", additional_kwargs={'tool_calls': [{'id': '8rymcepcy', 'function': {'arguments': '{"name":"priya"}', 'name': 'get_resume'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 486, 'total_tokens': 542, 'completion_time': 0.126326858, 'completion_tokens_details': None, 'prompt_time': 0.034402934, 'prompt_tokens_details': None, 'queue_time': 0.049912005, 'total_time': 0.160729792}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0fbc5-f626-7152-b392-3a589a036f92-0', tool_calls=[{'name': 'get_resume', 'args': {'name': 'priya'}, 'id': '8rymcepcy', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 486, 'output_tokens':

In [56]:
result.additional_kwargs

{'tool_calls': [{'id': '8rymcepcy',
   'function': {'arguments': '{"name":"priya"}', 'name': 'get_resume'},
   'type': 'function'}]}

In [55]:
result.additional_kwargs['tool_calls']

[{'id': '8rymcepcy',
  'function': {'arguments': '{"name":"priya"}', 'name': 'get_resume'},
  'type': 'function'}]

## Perfect Way / structure way

In [57]:
llm = init_chat_model(MODEL , temperature=0)
llm_with_tools = llm.bind_tools(TOOLS)

In [58]:
question = 'can we select priya for Python developer interview'
messages =[HumanMessage(content=question)]
print(f"[bold cyan] USER : [/bold cyan] {question} \n")

 USER :  can we select priya for Python developer interview

In [59]:
messages

[HumanMessage(content='can we select priya for Python developer interview', additional_kwargs={}, response_metadata={})]

## LLM 

In [60]:
ai = llm_with_tools.invoke(messages)

In [61]:
ai

AIMessage(content="I don't have any information about a candidate named Priya in my available tools. Let me check if there's a resume available for her.\n\n", additional_kwargs={'tool_calls': [{'id': 'hb45f91cm', 'function': {'arguments': '{"name":"priya"}', 'name': 'get_resume'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 486, 'total_tokens': 542, 'completion_time': 0.13064633, 'completion_tokens_details': None, 'prompt_time': 0.041883564, 'prompt_tokens_details': None, 'queue_time': 0.071137914, 'total_time': 0.172529894}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_4560dae850', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0fbc9-945f-74e3-9cb1-710c094c8142-0', tool_calls=[{'name': 'get_resume', 'args': {'name': 'priya'}, 'id': 'hb45f91cm', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 486, 'output_tokens': 

In [62]:
messages.append(ai)

In [63]:
messages

[HumanMessage(content='can we select priya for Python developer interview', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I don't have any information about a candidate named Priya in my available tools. Let me check if there's a resume available for her.\n\n", additional_kwargs={'tool_calls': [{'id': 'hb45f91cm', 'function': {'arguments': '{"name":"priya"}', 'name': 'get_resume'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 486, 'total_tokens': 542, 'completion_time': 0.13064633, 'completion_tokens_details': None, 'prompt_time': 0.041883564, 'prompt_tokens_details': None, 'queue_time': 0.071137914, 'total_time': 0.172529894}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_4560dae850', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0fbc9-945f-74e3-9cb1-710c094c8142-0', tool_calls=[{'name': 'get_resume', 'args': {'name': 'priya'

In [64]:
ai.tool_calls

[{'name': 'get_resume',
  'args': {'name': 'priya'},
  'id': 'hb45f91cm',
  'type': 'tool_call'}]

In [65]:
ai.content

"I don't have any information about a candidate named Priya in my available tools. Let me check if there's a resume available for her.\n\n"

### Tools or LLM ko call karne par .invoke() function call karna pdta hai

In [75]:
tool_name = ai.tool_calls[0]["name"]
tool_arg = ai.tool_calls[0]["args"]["name"]
tool_id = ai.tool_calls[0]["id"]
resume = get_resume.invoke(tool_arg)
print(resume)

Name: Priya Singh
Role: DevOps Engineer
Experience: 5 years
Skills: AWS, Kubernetes, Docker, Terraform, CI/CD (GitHub Actions), Linux
Background:
5 years of hands-on experience in cloud infrastructure and automating deployment pipelines. Expert in AWS services,
managing Kubernetes clusters, and infrastructure-as-code with Terraform.

In [76]:
tool_name

'get_resume'

In [77]:
tool_arg 

'priya'

In [78]:
tool_id

'hb45f91cm'

## llm ko dena hai

In [79]:
messages.append(ToolMessage(content=resume,tool_call_id=tool_id))

In [80]:
messages

[HumanMessage(content='can we select priya for Python developer interview', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I don't have any information about a candidate named Priya in my available tools. Let me check if there's a resume available for her.\n\n", additional_kwargs={'tool_calls': [{'id': 'hb45f91cm', 'function': {'arguments': '{"name":"priya"}', 'name': 'get_resume'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 486, 'total_tokens': 542, 'completion_time': 0.13064633, 'completion_tokens_details': None, 'prompt_time': 0.041883564, 'prompt_tokens_details': None, 'queue_time': 0.071137914, 'total_time': 0.172529894}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_4560dae850', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0fbc9-945f-74e3-9cb1-710c094c8142-0', tool_calls=[{'name': 'get_resume', 'args': {'name': 'priya'

In [81]:
result_final = llm_with_tools.invoke(messages) 

In [82]:
result_final

AIMessage(content="Based on Priya Singh's resume, I would **not recommend** selecting her for a **Python Developer** interview. Here's why:\n\n**Priya's Profile:**\n- **Role:** DevOps Engineer\n- **Experience:** 5 years\n- **Skills:** AWS, Kubernetes, Docker, Terraform, CI/CD (GitHub Actions), Linux\n- **Background:** Cloud infrastructure and deployment pipeline automation\n\n**Why she's not a fit for a Python Developer role:**\n- Her expertise is in **DevOps/Cloud Infrastructure**, not application development\n- Her skills focus on **infrastructure-as-code, containerization, and deployment automation**\n- There's **no mention of Python** or any application-level programming skills\n- Her background is in **operations and infrastructure**, not software development\n\n**Recommendation:**\n- If you're looking for a **DevOps Engineer**, Priya would be an excellent candidate\n- For a **Python Developer** role, you'd want someone with experience in Python frameworks (Django, Flask, FastAPI)

In [84]:
print(result_final.content)

Based on Priya Singh's resume, I would **not recommend** selecting her for a **Python Developer** interview. Here's
why:

**Priya's Profile:**
- **Role:** DevOps Engineer
- **Experience:** 5 years
- **Skills:** AWS, Kubernetes, Docker, Terraform, CI/CD (GitHub Actions), Linux
- **Background:** Cloud infrastructure and deployment pipeline automation

**Why she's not a fit for a Python Developer role:**
- Her expertise is in **DevOps/Cloud Infrastructure**, not application development
- Her skills focus on **infrastructure-as-code, containerization, and deployment automation**
- There's **no mention of Python** or any application-level programming skills
- Her background is in **operations and infrastructure**, not software development

**Recommendation:**
- If you're looking for a **DevOps Engineer**, Priya would be an excellent candidate
- For a **Python Developer** role, you'd want someone with experience in Python frameworks (Django, Flask, 
FastAPI), data structures, algorithms, and application development

Would you like me to help with anything else?